## Importando e compreendendo os dados como vieram

In [39]:
import pandas as pd
import json
from pathlib import Path

In [40]:
#padronizando caminhos

BASE_DIR = Path.cwd().parent
DATA_RAW = BASE_DIR / "datasets" / "dados_recebidos"

produtos_path = DATA_RAW / "produtos_raw.csv"
vendas_path = DATA_RAW / "vendas_2023_2024.csv"
clientes_path = DATA_RAW / "clientes_crm.json"
custos_path = DATA_RAW / "custos_importacao.json"

df_produtos = pd.read_csv(produtos_path)
df_vendas = pd.read_csv(vendas_path)

with open(clientes_path, "r", encoding="utf-8") as f:
    clientes_data = json.load(f)
df_clientes = pd.DataFrame(clientes_data)

with open(custos_path, "r", encoding="utf-8") as f:
    custos_data = json.load(f)
df_custos = pd.DataFrame(custos_data)

## Primeira olhada nas bases

Conferindo o tamanho de cada base e olhando as primeiras linhas para entender melhor a estrutura dos dados.

Aqui a ideia é identificar rapidamente se as colunas fazem sentido, se existe algum problema de formatação e se já aparece alguma inconsistência logo no começo.

In [41]:
print("Produtos:", df_produtos.shape)
print("Vendas:", df_vendas.shape)
print("Clientes:", df_clientes.shape)
print("Custos:", df_custos.shape)

display(df_produtos.head())
display(df_vendas.head())
display(df_clientes.head())
display(df_custos.head())

df_produtos.info()
df_vendas.info()
df_clientes.info()
df_custos.info()

Produtos: (157, 4)
Vendas: (9895, 6)
Clientes: (49, 4)
Custos: (150, 4)


,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


,full_name,location,code,email
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro#gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira#gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas#icloud.com


,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105..."
1,2,Transponder Furuno Marlin,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432..."
2,3,Radar Furuno Pulse Leviathan,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254..."
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909..."
4,5,Piloto Automático Furuno Storm,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600..."


<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   name             157 non-null    str  
 1   price            157 non-null    str  
 2   code             157 non-null    int64
 3   actual_category  157 non-null    str  
dtypes: int64(1), str(3)
memory usage: 5.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          9895 non-null   int64  
 1   id_client   9895 non-null   int64  
 2   id_product  9895 non-null   int64  
 3   qtd         9895 non-null   int64  
 4   total       9895 non-null   float64
 5   sale_date   9895 non-null   str    
dtypes: float64(1), int64(4), str(1)
memory usage: 464.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 4 columns):
 #   Column     Non-Nul

## Problemas na base de produtos

Logo de cara deu pra ver que a coluna de categoria está bem bagunçada.

Tem a mesma categoria escrita de várias formas diferentes, por exemplo:
- ELETRONICOS
- E L E T R Ô N I C O S
- Eletrunicos
- Eletronicz

Isso pode dar problema quando for agrupar ou analisar os dados depois.

Outro ponto é que o preço está como texto, então do jeito que está não dá pra fazer cálculo. Vou precisar converter isso pra número.

## Problemas na base de vendas

A base está mais organizada, mas a coluna de data não segue um padrão único.

Tem datas no formato:
- 2023-09-10  
- 15-09-2024  

Isso pode dar problema na hora de analisar por período, então vou padronizar depois.

## Problemas na base de clientes

Aqui já aparecem alguns problemas mais claros.

Os e-mails não estão todos corretos. Em alguns casos aparece "#" no lugar de "@", o que invalida o contato.

A localização também está bem inconsistente. Cada registro usa um formato diferente, como:
- PE , Recife  
- Rio Grande,RS  
- PA - Santarém Novo  

Do jeito que está, fica difícil usar essa informação pra qualquer análise por região.

Vou precisar organizar isso melhor.

## Problemas na base de custos

Essa base tem um campo com uma estrutura mais complexa (uma lista com histórico de dados).

Do jeito que está, não é tão simples de usar direto. Provavelmente vou precisar tratar isso depois pra facilitar as análises.

In [42]:
df_produtos["actual_category"].value_counts()

actual_category
AncorageM                9
Propução                 8
Ancoraguem               8
Eletronicoz              7
eletrônicos              7
ELETRONICOS              6
E L E T R Ô N I C O S    6
PROPULSAO                6
P R O P U L S Ã O        6
propulsão                6
Eletrunicos              5
eLeTrÔnIcOs              5
Propulção                5
Prop                     5
Propulssão               5
Encoragem                5
Ancorajm                 5
A N C O R A G E M        5
aNcOrAgEm                5
Eletrônicos              4
propulsao                4
eletronicos              3
EletrônicoS              3
Propulçao                3
Ancoragem                3
Ancorajem                3
Eletroniscos             2
Eletronicos              2
pRoPuLsÃo                2
Propulsam                2
AnCoRaGeM                2
ancoragem                2
Ancorajen                2
ELEtRÔNICOS              1
PrOpUlSãO                1
ANCORAGEM                1
Encoragi    

In [43]:
#Padronizando textos

import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return texto
    
    texto = texto.upper()
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    texto = texto.replace(" ", "")
    
    return texto

df_produtos["categoria_norm"] = df_produtos["actual_category"].apply(normalizar_texto)

def corrigir_categoria(cat):
    if "ELETRONIC" in cat:
        return "ELETRONICOS"
    elif "PROPUL" in cat:
        return "PROPULSAO"
    elif "ANCOR" in cat:
        return "ANCORAGEM"
    else:
        return cat

df_produtos["categoria_final"] = df_produtos["categoria_norm"].apply(corrigir_categoria)

In [44]:
df_produtos["categoria_final"].value_counts()

categoria_final
ANCORAGEM       47
ELETRONICOS     44
PROPULSAO       40
PROPUCAO         8
ELETRUNICOS      5
PROP             5
ENCORAGEM        5
ELETRONISCOS     2
ENCORAGI         1
Name: count, dtype: int64

## Refinando a padronização das categorias

Depois da primeira limpeza, ainda sobraram algumas variações com erro de digitação ou abreviação.

Então aqui vou fazer um segundo ajuste, agora mais direcionado, para consolidar tudo nas três categorias corretas.

In [45]:
def corrigir_categoria_final(cat):
    if cat in ["ELETRONICOS", "ELETRUNICOS", "ELETRONISCOS"]:
        return "ELETRONICOS"
    elif cat in ["PROPULSAO", "PROPUCAO", "PROP"]:
        return "PROPULSAO"
    elif cat in ["ANCORAGEM", "ENCORAGEM", "ENCORAGI"]:
        return "ANCORAGEM"
    else:
        return cat

df_produtos["categoria_final"] = df_produtos["categoria_final"].apply(corrigir_categoria_final)

df_produtos["categoria_final"].value_counts()

categoria_final
PROPULSAO      53
ANCORAGEM      53
ELETRONICOS    51
Name: count, dtype: int64

In [46]:
#Conferência

df_produtos["actual_category"] = df_produtos["categoria_final"]
df_produtos.drop(columns=["categoria_norm", "categoria_final"], inplace=True)

df_produtos.head()

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,ELETRONICOS
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,ELETRONICOS
4,Piloto Automático Furuno Storm,R$ 23669.01,5,ELETRONICOS


## Tratamento da base de clientes

Depois de ajustar a base de produtos, parti para a base de clientes.

Aqui os principais problemas aparecem nos campos de e-mail e localização, então comecei olhando esses dois pontos com mais atenção.

In [47]:
df_clientes.head(10)
df_clientes["location"].value_counts()
df_clientes["email"].head(15)

0                 femininos.oliveira.antunes@icloud.com
1      nunes.fernanda.soares.azevedo.vieira@outlook.com
2              farias.teixeira.daniel.ribeiro#gmail.com
3                              thiago.moreira#gmail.com
4                              pedro.freitas#icloud.com
5     coelho.pinheiro.peixoto.antônia.cavalcanti@aol...
6           torres.barros.rocha.bianca.siqueira#aol.com
7                       pimentel.alves.luiz#outlook.com
8                 lucas.lopes.guedes.cunha#tutanota.com
9                                  paiva.débora#gmx.com
10                     torres.monteiro.victor@gmail.com
11                       rafael.pereira.barros#zoho.com
12                 martins.guimarães.carlos#hotmail.com
13              vieira.silva.amaral.gabriela@icloud.com
14            lopes.alves.pacheco.rocha.carla#yahoo.com
Name: email, dtype: str

## Ajustando os e-mails

A primeira correção mais direta na base de clientes é o campo de e-mail.

Em alguns registros, o caractere `#` foi usado no lugar de `@`, então vou corrigir isso antes de validar os endereços.

In [48]:
def corrigir_email(email):
    if pd.isna(email):
        return email
    return email.strip().lower().replace("#", "@")

df_clientes["email_corrigido"] = df_clientes["email"].apply(corrigir_email)

df_clientes[["email", "email_corrigido"]].head(15)

,email,email_corrigido
0,femininos.oliveira.antunes@icloud.com,femininos.oliveira.antunes@icloud.com
1,nunes.fernanda.soares.azevedo.vieira@outlook.com,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,farias.teixeira.daniel.ribeiro#gmail.com,farias.teixeira.daniel.ribeiro@gmail.com
3,thiago.moreira#gmail.com,thiago.moreira@gmail.com
4,pedro.freitas#icloud.com,pedro.freitas@icloud.com
5,coelho.pinheiro.peixoto.antônia.cavalcanti@aol...,coelho.pinheiro.peixoto.antônia.cavalcanti@aol...
6,torres.barros.rocha.bianca.siqueira#aol.com,torres.barros.rocha.bianca.siqueira@aol.com
7,pimentel.alves.luiz#outlook.com,pimentel.alves.luiz@outlook.com
8,lucas.lopes.guedes.cunha#tutanota.com,lucas.lopes.guedes.cunha@tutanota.com
9,paiva.débora#gmx.com,paiva.débora@gmx.com


In [49]:
import re

def email_valido(email):
    if pd.isna(email):
        return False
    padrao = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    return bool(re.match(padrao, email))

df_clientes["email_valido"] = df_clientes["email_corrigido"].apply(email_valido)

df_clientes["email_valido"].value_counts()

email_valido
True    49
Name: count, dtype: int64

In [50]:
df_clientes["location"].value_counts()

location
BA - Porto Seguro                 2
Aratu (Candeias) , BA             1
PE , Recife                       1
Rio Grande,RS                     1
AC , Rio Branco                   1
PA - Santarém Novo                1
Fortaleza do Tabocão , TO         1
PB/Cabedelo                       1
SE - Aracaju                      1
PB - João Pessoa                  1
Santarém / PA                     1
TO , Fortaleza do Tabocão         1
PA / Santarém                     1
AM , Itacoatiara                  1
Fortaleza do Tabocão,TO           1
Fortaleza,CE                      1
MS - Corumbá                      1
Santarém - PA                     1
Maceió / AL                       1
PA , Santarém Novo                1
AC,Rio Branco                     1
SE / Aracaju                      1
Santos - SP                       1
Laguna / SC                       1
ES / São Mateus                   1
Manaus/AM                         1
Salvador,BA                       1
PR , Antonina      

## Problemas na localização

O campo de localização está bem inconsistente.

Existem diferentes formas de separar cidade e estado (vírgula, hífen, barra) e também variação na ordem das informações.

Isso dificulta qualquer análise por região, então o próximo passo é padronizar esse campo antes de separar cidade e UF.

In [51]:
def padronizar_location(loc):
    if pd.isna(loc):
        return None
    
    loc = loc.upper().strip()
    
    # padronizar separadores
    loc = re.sub(r"\s*[-/]\s*", "-", loc)
    loc = re.sub(r"\s*,\s*", "- ", loc)
    
    return loc

df_clientes["location_padronizada"] = df_clientes["location"].apply(padronizar_location)

df_clientes[["location", "location_padronizada"]].head(10)

,location,location_padronizada
0,"Aratu (Candeias) , BA",ARATU (CANDEIAS)- BA
1,"PE , Recife",PE- RECIFE
2,"Rio Grande,RS",RIO GRANDE- RS
3,"AC , Rio Branco",AC- RIO BRANCO
4,PA - Santarém Novo,PA-SANTARÉM NOVO
5,"Fortaleza do Tabocão , TO",FORTALEZA DO TABOCÃO- TO
6,PB/Cabedelo,PB-CABEDELO
7,SE - Aracaju,SE-ARACAJU
8,PB - João Pessoa,PB-JOÃO PESSOA
9,Santarém / PA,SANTARÉM-PA


In [52]:
ufs = {
    "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG",
    "PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"
}

def extrair_cidade_uf(loc):
    if pd.isna(loc):
        return pd.Series([None, None])
    
    partes = [p.strip() for p in loc.split("-")]
    
    cidade = None
    uf = None
    
    for p in partes:
        if p in ufs:
            uf = p
    
    partes_sem_uf = [p for p in partes if p not in ufs]
    
    if partes_sem_uf:
        cidade = partes_sem_uf[-1]
    
    return pd.Series([cidade, uf])

df_clientes[["cidade", "uf"]] = df_clientes["location_padronizada"].apply(extrair_cidade_uf)

df_clientes[["location_padronizada", "cidade", "uf"]].head(15)

,location_padronizada,cidade,uf
0,ARATU (CANDEIAS)- BA,ARATU (CANDEIAS),BA
1,PE- RECIFE,RECIFE,PE
2,RIO GRANDE- RS,RIO GRANDE,RS
3,AC- RIO BRANCO,RIO BRANCO,AC
4,PA-SANTARÉM NOVO,SANTARÉM NOVO,PA
5,FORTALEZA DO TABOCÃO- TO,FORTALEZA DO TABOCÃO,TO
6,PB-CABEDELO,CABEDELO,PB
7,SE-ARACAJU,ARACAJU,SE
8,PB-JOÃO PESSOA,JOÃO PESSOA,PB
9,SANTARÉM-PA,SANTARÉM,PA


## Resultado do tratamento de localização

Após padronizar os separadores e organizar o formato das strings, foi possível extrair corretamente a cidade e a UF.

Mesmo com variações na ordem e nos formatos originais, os dados foram estruturados de forma consistente, permitindo análises por região com mais segurança.

In [53]:
df_clientes["email"] = df_clientes["email_corrigido"]
df_clientes["location"] = df_clientes["location_padronizada"]

df_clientes.drop(columns=["email_corrigido", "location_padronizada"], inplace=True)

df_clientes.head()

,full_name,location,code,email,email_valido,cidade,uf
0,Femininos Oliveira Antunes,ARATU (CANDEIAS)- BA,1,femininos.oliveira.antunes@icloud.com,True,ARATU (CANDEIAS),BA
1,Fernanda Azevedo Soares Nunes Vieira,PE- RECIFE,2,nunes.fernanda.soares.azevedo.vieira@outlook.com,True,RECIFE,PE
2,Daniel Farias Ribeiro Teixeira,RIO GRANDE- RS,3,farias.teixeira.daniel.ribeiro@gmail.com,True,RIO GRANDE,RS
3,Thiago Moreira,AC- RIO BRANCO,4,thiago.moreira@gmail.com,True,RIO BRANCO,AC
4,Pedro Freitas,PA-SANTARÉM NOVO,5,pedro.freitas@icloud.com,True,SANTARÉM NOVO,PA


In [ ]:
df_vendas["sale_date"].head(10)

0   2023-10-09
1          NaT
2          NaT
3   2023-03-02
4   2024-12-02
5          NaT
6          NaT
7          NaT
8          NaT
9   2023-07-05
Name: sale_date, dtype: datetime64[us]

In [60]:
# primeira tentativa: formato ano-mes-dia
df_vendas["sale_date_tratada"] = pd.to_datetime(
    df_vendas["sale_date_original"],
    format="%Y-%m-%d",
    errors="coerce"
)

# segunda tentativa: formato dia-mes-ano para o que ainda ficou vazio
mask = df_vendas["sale_date_tratada"].isna()

df_vendas.loc[mask, "sale_date_tratada"] = pd.to_datetime(
    df_vendas.loc[mask, "sale_date_original"],
    format="%d-%m-%Y",
    errors="coerce"
)

# conferência
df_vendas[["sale_date_original", "sale_date_tratada"]].head(10)

,sale_date_original,sale_date_tratada
0,2023-09-10,2023-09-10
1,15-09-2024,2024-09-15
2,2024-08-13,2024-08-13
3,2023-02-03,2023-02-03
4,2024-02-12,2024-02-12
5,2023-09-26,2023-09-26
6,2024-02-28,2024-02-28
7,07-11-2023,2023-11-07
8,2024-08-25,2024-08-25
9,2023-05-07,2023-05-07


In [61]:
df_vendas["sale_date_tratada"].isna().sum()

np.int64(0)

In [62]:
df_vendas["sale_date"] = df_vendas["sale_date_tratada"]
df_vendas.drop(columns=["sale_date_original", "sale_date_tratada", "sale_date_formatada"], errors="ignore", inplace=True)